# MedIQ — TFT Resume Training
## Continues from `epoch_8-step_81171.ckpt`

### Two modes in this notebook
- **Mode A — Resume from checkpoint**: continues training from epoch 8 → 29. 
  Use this if you just want to finish the original run.
- **Mode B — Fast retrain**: higher batch size + fewer epochs, targets 2–4h on a free T4.
  Use this if you want a fresh faster run.

**Set `MODE = 'resume'` or `MODE = 'fast'` in the config cell.**

In [ ]:
!pip install -q pytorch-forecasting lightning shap lightgbm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## CONFIG — Edit these paths and choose your mode

In [ ]:
import os

# ── Paths ─────────────────────────────────────────────────────────────────────
ZIP_PATH      = '/content/drive/MyDrive/mediq/datasets/training_setA.zip'
EXTRACT_PATH  = '/content/training_setA/'
SAVE_DIR      = '/content/drive/MyDrive/mediq/mediq_sepsis_artifacts/'

# Path to the checkpoint file you uploaded (epoch_8-step_81171.ckpt).
# Upload it to Drive and point here — OR upload directly to Colab (/content/).
CKPT_PATH = '/content/drive/MyDrive/mediq/checkpoints/epoch_8-step_81171.ckpt'
# If you uploaded directly to Colab (not Drive), use:
# CKPT_PATH = '/content/epoch_8-step_81171.ckpt'

os.makedirs(SAVE_DIR, exist_ok=True)

# ── Mode ──────────────────────────────────────────────────────────────────────
# 'resume' : pick up from epoch 8, finish the original 30-epoch run
# 'fast'   : fresh start with optimised settings, done in 2-4h
MODE = 'fast'

# ── Fast-mode settings (only used when MODE = 'fast') ─────────────────────────
FAST_BATCH_SIZE  = 512   # 4× bigger than original 128 — T4 handles this fine
FAST_MAX_EPOCHS  = 15    # 15 epochs at 4× batch → roughly same compute as 7 original epochs
FAST_LR          = 0.05  # slightly higher LR to compensate for larger batches (linear scaling)

print(f'Mode: {MODE}')
print(f'Checkpoint: {CKPT_PATH}')

## 1. Extract Data

In [ ]:
if not os.path.exists(EXTRACT_PATH):
    print('Unzipping...')
    os.makedirs(EXTRACT_PATH, exist_ok=True)
    os.system(f'unzip -q "{ZIP_PATH}" -d "{EXTRACT_PATH}"')
else:
    print('Already extracted.')

files = []
for root, dirs, filenames in os.walk(EXTRACT_PATH):
    psv = [f for f in filenames if f.endswith('.psv')]
    if psv:
        EXTRACT_PATH = root + '/'
        files = psv
        break
print(f'Total patient files: {len(files)}')

## 2. Preprocessing
Identical to original notebook — do not change.

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

CORE_VITALS = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp']
LAB_VALUES  = ['Lactate', 'WBC', 'Creatinine', 'BUN', 'Glucose',
               'Potassium', 'Hgb', 'Platelets', 'Bilirubin_total']
STATIC_COLS = ['Age', 'Gender']

def load_patients(extract_path):
    psv_files = sorted([f for f in os.listdir(extract_path) if f.endswith('.psv')])
    all_dfs = []
    for fname in tqdm(psv_files, desc='Loading patients'):
        p = pd.read_csv(os.path.join(extract_path, fname), sep='|')
        p['patient_id'] = fname.replace('.psv', '')
        all_dfs.append(p)
    return pd.concat(all_dfs, ignore_index=True)

def select_features(df, missing_threshold=80.0):
    candidates = CORE_VITALS + LAB_VALUES
    missing_pct = df[[c for c in candidates if c in df.columns]].isnull().mean() * 100
    return [c for c in candidates if c in df.columns and missing_pct[c] < missing_threshold]

def impute(df, feature_cols):
    df = df.sort_values(['patient_id', 'ICULOS']).copy()
    df[feature_cols] = df.groupby('patient_id')[feature_cols].ffill()
    medians = df[feature_cols].median()
    df[feature_cols] = df[feature_cols].fillna(medians)
    return df, medians

def add_comorbidity_flags(df):
    if 'Glucose' in df.columns:
        pat_glucose = df.groupby('patient_id')['Glucose'].mean()
        diabetic_ids = pat_glucose[pat_glucose > 140].index
        df['is_diabetic'] = df['patient_id'].isin(diabetic_ids).astype(int)
    else:
        df['is_diabetic'] = 0
    df['is_elderly'] = (df.get('Age', 0) > 65).astype(int)
    return df

def build_frame(extract_path, missing_threshold=80.0):
    df = load_patients(extract_path)
    feature_cols = select_features(df, missing_threshold)
    df, medians = impute(df, feature_cols)
    df = add_comorbidity_flags(df)
    df['time_idx']    = df.groupby('patient_id').cumcount()
    df['patient_id']  = df['patient_id'].astype(str)
    df['Age']         = df.get('Age', pd.Series(60, index=df.index)).fillna(60)
    df['Gender']      = df.get('Gender', pd.Series(0, index=df.index)).fillna(0).astype(str)
    df['is_diabetic'] = df['is_diabetic'].astype(str)
    df['is_elderly']  = df['is_elderly'].astype(str)
    return df, feature_cols, medians

df, feature_cols, medians = build_frame(EXTRACT_PATH)
print(f'Patients: {df["patient_id"].nunique():,} | Rows: {len(df):,}')
print(f'Features ({len(feature_cols)}): {feature_cols}')

## 3. Build Dataset
Same dataset definition as original — **must be identical** when resuming, so the model architecture matches the checkpoint.

In [ ]:
import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss

MAX_ENCODER_LENGTH    = 24
MAX_PREDICTION_LENGTH = 6

training_cutoff = df['time_idx'].max() - MAX_PREDICTION_LENGTH

training_ds = TimeSeriesDataSet(
    df[lambda x: x.time_idx <= training_cutoff],
    time_idx='time_idx',
    target='SepsisLabel',
    group_ids=['patient_id'],
    max_encoder_length=MAX_ENCODER_LENGTH,
    min_encoder_length=4,
    max_prediction_length=MAX_PREDICTION_LENGTH,
    min_prediction_length=1,
    static_categoricals=['Gender', 'is_diabetic', 'is_elderly'],
    static_reals=['Age'],
    time_varying_known_reals=['time_idx'],
    time_varying_unknown_reals=feature_cols,
    target_normalizer=None,
    add_relative_time_idx=True,
    add_target_scales=False,
    add_encoder_length=True,
    allow_missing_timesteps=True,
)

validation_ds = TimeSeriesDataSet.from_dataset(
    training_ds, df, predict=True, stop_randomization=True
)

print(f'Training samples : {len(training_ds):,}')
print(f'Validation samples: {len(validation_ds):,}')

## 4A — RESUME from epoch 8 checkpoint
Skips if `MODE != 'resume'`.

**How it works:**  
`trainer.fit(..., ckpt_path=CKPT_PATH)` tells Lightning to restore:
- Model weights exactly as saved at end of epoch 8
- Optimizer state (Adam momentum buffers) — crucial, otherwise LR warmup restarts
- `trainer.current_epoch` counter → starts at 9, counts up to `max_epochs=30`
- `global_step` counter → EarlyStopping `patience` tracking continues correctly

Nothing else changes. Same architecture, same dataset, same callbacks.

In [ ]:
if MODE == 'resume':

    BATCH_SIZE = 128   # same as original

    train_loader = training_ds.to_dataloader(
        train=True, batch_size=BATCH_SIZE, num_workers=2  # num_workers=2 speeds up data loading vs 0
    )
    val_loader = validation_ds.to_dataloader(
        train=False, batch_size=BATCH_SIZE, num_workers=2
    )

    # ── Model: same architecture as original — do not change any of these values ──
    tft = TemporalFusionTransformer.from_dataset(
        training_ds,
        learning_rate=0.03,
        hidden_size=32,
        attention_head_size=4,
        dropout=0.2,
        hidden_continuous_size=16,
        loss=QuantileLoss(),
        optimizer='adam',
        log_interval=10,
    )
    print(f'Model params: {sum(p.numel() for p in tft.parameters()):,}')

    early_stop = EarlyStopping(monitor='val_loss', patience=5, mode='min')

    # Save a checkpoint after every epoch so you don't lose progress again
    checkpoint_cb = ModelCheckpoint(
        dirpath=SAVE_DIR + 'checkpoints/',
        filename='tft-{epoch:02d}-{val_loss:.4f}',
        save_top_k=3,          # keep 3 best
        monitor='val_loss',
        mode='min',
        save_last=True,        # always keep a 'last.ckpt' too
    )

    trainer = pl.Trainer(
        max_epochs=30,          # same as original — Lightning resumes from epoch 9
        accelerator='auto',
        gradient_clip_val=0.1,
        callbacks=[early_stop, checkpoint_cb],
        default_root_dir=SAVE_DIR + 'logs/',
        precision='16-mixed',   # FP16 on T4 — cuts VRAM in half, ~1.5× faster, no accuracy loss
    )

    print('Resuming from:', CKPT_PATH)
    print('Will train epochs 9 → 29 (or until EarlyStopping triggers)')

    # ── THE KEY LINE ──────────────────────────────────────────────────────────
    # ckpt_path restores weights + optimizer state + epoch counter.
    # This is NOT the same as load_from_checkpoint() which only loads weights.
    trainer.fit(tft, train_dataloaders=train_loader, val_dataloaders=val_loader,
                ckpt_path=CKPT_PATH)

    # Save final model
    trainer.save_checkpoint(SAVE_DIR + 'sepsis_tft_resumed_final.ckpt')
    print('Saved final model to:', SAVE_DIR + 'sepsis_tft_resumed_final.ckpt')
    print(f'Best model: {checkpoint_cb.best_model_path}')

else:
    print('Skipping resume cell (MODE =', MODE, ')')

## 4B — FAST retrain (2–4 hours on T4)
Skips if `MODE != 'fast'`.

**What changes and why:**

| Setting | Original | Fast | Reason |
|---|---|---|---|
| `batch_size` | 128 | 512 | 4× bigger batches → 4× fewer gradient steps per epoch |
| `max_epochs` | 30 | 15 | Fewer epochs needed because each batch sees more data |
| `learning_rate` | 0.03 | 0.05 | Linear scaling rule: larger batch → larger LR (keeps convergence stable) |
| `num_workers` | 0 | 2 | Overlaps data loading with GPU compute |
| `precision` | 32 | 16-mixed | FP16 on T4 halves VRAM, ~1.5× faster |
| Architecture | — | unchanged | hidden_size, dropout etc must stay the same for inference.py compatibility |

In [ ]:
if MODE == 'fast':

    train_loader = training_ds.to_dataloader(
        train=True, batch_size=FAST_BATCH_SIZE, num_workers=2
    )
    val_loader = validation_ds.to_dataloader(
        train=False, batch_size=FAST_BATCH_SIZE, num_workers=2
    )

    # Architecture is IDENTICAL to original — do not change hidden_size etc.
    # Only LR changes (to go with the bigger batch).
    tft = TemporalFusionTransformer.from_dataset(
        training_ds,
        learning_rate=FAST_LR,
        hidden_size=32,
        attention_head_size=4,
        dropout=0.2,
        hidden_continuous_size=16,
        loss=QuantileLoss(),
        optimizer='adam',
        log_interval=10,
    )
    print(f'Model params: {sum(p.numel() for p in tft.parameters()):,}')

    early_stop = EarlyStopping(monitor='val_loss', patience=4, mode='min')  # patience=4 (was 5) since epochs are shorter

    checkpoint_cb = ModelCheckpoint(
        dirpath=SAVE_DIR + 'checkpoints_fast/',
        filename='tft-fast-{epoch:02d}-{val_loss:.4f}',
        save_top_k=3,
        monitor='val_loss',
        mode='min',
        save_last=True,
    )

    trainer = pl.Trainer(
        max_epochs=FAST_MAX_EPOCHS,
        accelerator='auto',
        gradient_clip_val=0.1,
        callbacks=[early_stop, checkpoint_cb],
        default_root_dir=SAVE_DIR + 'logs_fast/',
        precision='16-mixed',
    )

    print(f'Fast mode: batch={FAST_BATCH_SIZE}, epochs={FAST_MAX_EPOCHS}, lr={FAST_LR}')
    trainer.fit(tft, train_dataloaders=train_loader, val_dataloaders=val_loader)

    trainer.save_checkpoint(SAVE_DIR + 'sepsis_tft_fast_final.ckpt')
    print('Saved:', SAVE_DIR + 'sepsis_tft_fast_final.ckpt')
    print(f'Best: {checkpoint_cb.best_model_path}')

else:
    print('Skipping fast cell (MODE =', MODE, ')')

## 5. Evaluate
Runs after either mode. Reports AUROC and confusion matrix at calibrated thresholds.

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# Use whichever model just finished training
eval_loader = validation_ds.to_dataloader(train=False, batch_size=128, num_workers=0)

raw_preds   = tft.predict(eval_loader, mode='quantiles', return_x=True, return_y=True)
quantiles   = raw_preds.output.numpy()
median_idx  = quantiles.shape[-1] // 2
pred_risk   = quantiles[:, 0, median_idx]
true_label  = raw_preds.y[0].numpy()[:, 0]

auroc = roc_auc_score(true_label, pred_risk)
print(f'AUROC: {auroc:.3f}')

pred_binary = (pred_risk >= 0.65).astype(int)
cm = confusion_matrix(true_label, pred_binary)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Sepsis', 'Sepsis'],
            yticklabels=['No Sepsis', 'Sepsis'])
ax.set_title(f'Confusion Matrix @ default threshold 0.65 (AUROC={auroc:.3f})')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.show()

print(classification_report(true_label, pred_binary, target_names=['No Sepsis', 'Sepsis']))

## 6. SHAP Surrogate
Identical to original notebook.

In [ ]:
import shap
import joblib
from lightgbm import LGBMClassifier

last_obs = df.sort_values('time_idx').groupby('patient_id').last()
X = last_obs[feature_cols].values
y = last_obs['SepsisLabel'].astype(int).values

surrogate = LGBMClassifier(n_estimators=100, max_depth=4, class_weight='balanced', verbosity=-1)
surrogate.fit(X, y)

explainer   = shap.TreeExplainer(surrogate)
shap_values = explainer.shap_values(X[:200])
shap.summary_plot(shap_values[1] if isinstance(shap_values, list) else shap_values,
                  X[:200], feature_names=feature_cols)

joblib.dump(surrogate, SAVE_DIR + 'shap_surrogate.pkl')
print('Saved surrogate to', SAVE_DIR + 'shap_surrogate.pkl')

# CRITICAL: Save dataset template so inference.py can use it
# This is needed for predict_trajectory() in inference.py
joblib.dump(training_ds, SAVE_DIR + 'dataset_template.pkl')
print('Saved dataset template to', SAVE_DIR + 'dataset_template.pkl')

## 7. Save Final Artifacts for Backend
Copies the best checkpoint to a stable path that `inference.py` expects.

In [ ]:
import shutil, json

# Copy best model to the stable path inference.py loads
best_path = checkpoint_cb.best_model_path
stable_path = SAVE_DIR + 'sepsis_tft.ckpt'

if best_path:
    shutil.copy2(best_path, stable_path)
    print(f'Best model copied to: {stable_path}')
    print(f'Best val_loss: {checkpoint_cb.best_model_score:.4f}')
else:
    trainer.save_checkpoint(stable_path)
    print(f'Final model saved to: {stable_path}')

# Save config so backend/inference.py has feature list
config = {
    'feature_cols': feature_cols,
    'max_encoder_length': MAX_ENCODER_LENGTH,
    'max_prediction_length': MAX_PREDICTION_LENGTH,
    'training_mode': MODE,
    'epochs_completed': int(trainer.current_epoch + 1),
    'auroc': float(auroc),
}
with open(SAVE_DIR + 'model_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('\n=== ARTIFACTS READY FOR BACKEND ===')
print('Location:', SAVE_DIR)
print('  sepsis_tft.ckpt       — TFT weights')
print('  shap_surrogate.pkl    — SHAP explainer')
print('  dataset_template.pkl  — TimeSeriesDataSet for inference.py')
print('  model_config.json     — feature list + metadata')
print(f'\nAUROC: {auroc:.3f}')
print('Download these 4 files and give to Anirudh.')